# Lab 06: ACID Transactions & Conflict Serializability

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

Precedence graph serializability verifier and atomic SQLite rollback under constraint checks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB06_ACID_Transactions_and_Serializability.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. ACID Atomicity & Conflict Serializability Precedence Graph

Two operations conflict if they belong to different transactions, access the same item, and at least one is a write. A schedule is conflict-serializable iff its precedence graph is acyclic.

In [1]:
import sqlite3


def is_conflict_serializable(ops):
    txns = sorted({t for t, _, _ in ops})
    adj = {t: set() for t in txns}
    for i in range(len(ops)):
        t1, op1, x1 = ops[i]
        for j in range(i + 1, len(ops)):
            t2, op2, x2 = ops[j]
            if t1 != t2 and x1 == x2 and (op1 == "W" or op2 == "W"):
                adj[t1].add(t2)
    indeg = {t: 0 for t in txns}
    for u in adj:
        for v in adj[u]:
            indeg[v] += 1
    q = [t for t in txns if indeg[t] == 0]
    order = []
    while q:
        u = q.pop(0)
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return len(order) == len(txns), order


s1 = [("T1", "R", "A"), ("T1", "W", "A"), ("T2", "R", "A"), ("T2", "W", "A")]
s2 = [("T1", "R", "A"), ("T2", "R", "A"), ("T1", "W", "A"), ("T2", "W", "A")]
print("Schedule 1 serializable:", is_conflict_serializable(s1))
print("Schedule 2 serializable:", is_conflict_serializable(s2))
assert is_conflict_serializable(s1)[0] and not is_conflict_serializable(s2)[0]


Schedule 1 serializable: (True, ['T1', 'T2'])
Schedule 2 serializable: (False, [])


## 2. Atomic Rollback under `CHECK (bal >= 0)` Constraint

Wrapping a multi-statement transfer inside `with conn:` rolls back partial updates automatically if any invariant fails.

In [2]:
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE Accounts (id TEXT PRIMARY KEY, bal INTEGER CHECK (bal >= 0))")
conn.execute("INSERT INTO Accounts VALUES ('Asha', 100), ('Omar', 50)")
conn.commit()

try:
    with conn:
        conn.execute("UPDATE Accounts SET bal = bal - 150 WHERE id = 'Asha'")
        conn.execute("UPDATE Accounts SET bal = bal + 150 WHERE id = 'Omar'")
except sqlite3.IntegrityError as exc:
    print("Rolled back failed overdraft:", exc)

balances = dict(conn.execute("SELECT id, bal FROM Accounts").fetchall())
print("Balances after rollback:", balances)
assert balances == {"Asha": 100, "Omar": 50}


Rolled back failed overdraft: CHECK constraint failed: bal >= 0
Balances after rollback: {'Asha': 100, 'Omar': 50}
